# Stable-DINO Smoke Train

Notebook tối giản để kiểm tra `Stable-DINO` có chạy được 1 iteration hay không.

In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "object_detection").exists():
    REPO_ROOT = REPO_ROOT.parent

assert (REPO_ROOT / "object_detection").exists(), f"Cannot find repo root from {Path.cwd()}"
os.chdir(REPO_ROOT)

print("repo_root:", REPO_ROOT)
print("python:", sys.executable)

In [ ]:
# Run this once on Kaggle if the current kernel does not have detectron2/detrex.
# After installation, restart the kernel, then run the notebook from the top again.
%pip install -q ninja yacs cython packaging
%pip install -q pycocotools shapely timm
%pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
%pip install -q 'git+https://github.com/IDEA-Research/detrex.git'


In [ ]:
import torch

print("torch:", torch.__version__)
print("mps_built:", torch.backends.mps.is_built())
print("mps_available:", torch.backends.mps.is_available())
print("cuda_available:", torch.cuda.is_available())

requested_device = "cuda"
resolved_device = "cuda" if torch.cuda.is_available() else "cpu"
print("requested_device:", requested_device)
print("resolved_device:", resolved_device)

In [ ]:
import importlib

required_modules = ["torch", "detectron2", "detrex"]
missing = []

for name in required_modules:
    try:
        importlib.import_module(name)
        print(f"ok: {name}")
    except Exception as exc:
        print(f"missing: {name} -> {exc}")
        missing.append(name)

assert not missing, f"Missing modules in current kernel: {missing}"


In [ ]:
DATASET = Path("/kaggle/input/datasets/vinhnqu/crack500-object-yolov12/crack500_det_files/crack500_det.yaml")
LOCAL_INIT_CKPT = Path.home() / ".torch" / "iopath_cache" / "detectron2" / "ImageNetPretrained" / "torchvision" / "R-50.pkl"
INIT_CKPT = str(LOCAL_INIT_CKPT) if LOCAL_INIT_CKPT.exists() else "detectron2://ImageNetPretrained/torchvision/R-50.pkl"
OUTPUT_DIR = REPO_ROOT / "object_detection" / "stable_dino" / "train" / "notebook_train_t4x2"
LOG_FILE = OUTPUT_DIR / "train_stdout.log"

print("dataset:", DATASET)
print("init_ckpt:", INIT_CKPT)
print("output_dir:", OUTPUT_DIR)
print("log_file:", LOG_FILE)

assert DATASET.exists(), DATASET

In [ ]:
import subprocess
from pathlib import Path

cmd = [
    sys.executable,
    "-m",
    "object_detection.stable_dino.train",
    "--dataset",
    str(DATASET),
    "--device",
    requested_device,
    "--num-gpus",
    "2",
    "--batch-size",
    "4",
    "--workers",
    "4",
    "--pin-memory",
    "--persistent-workers",
    "--prefetch-factor",
    "2",
    "--imgsz",
    "640",
    "--output-dir",
    str(OUTPUT_DIR),
    "--init-checkpoint",
    str(INIT_CKPT),
    "train.eval_period=5000",
    "train.log_period=20",
    "dataloader.test.num_workers=2",
]

print("python for training:", sys.executable)
print(" ".join(cmd))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
keep_markers = (
    "Starting training from iteration",
    "iter:",
    "Saving checkpoint",
    "Start inference on",
    "Evaluation results for bbox",
    "Total training time",
    "returncode:",
    "Traceback",
    "Error",
    "Exception",
)

with LOG_FILE.open("w", encoding="utf-8") as log_fp:
    process = subprocess.Popen(
        cmd,
        cwd=REPO_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        log_fp.write(line)
        if any(marker in line for marker in keep_markers):
            print(line, end="")
    result = process.wait()

print("returncode:", result)
print("full log saved to:", LOG_FILE)